# Databricks Auto Loader — Incremental File Ingestion

The same bronze ingestion as the Structured Streaming demo, but using **Auto Loader** (`cloudFiles`) for cloud-optimized incremental file discovery and schema evolution.

**Why Auto Loader over plain streaming:** it persists an evolving schema (`schemaLocation`) and uses efficient file discovery, so it scales to very high file volumes without re-listing the whole directory.

## 1. Read with Auto Loader
`cloudFiles` incrementally discovers new files, tracks a persistent `schemaLocation`, infers column types, and tolerates schema drift via `schemaHints`.

In [0]:
customers_df = (
                    spark.readStream
                         .format("cloudFiles")
                         .option("cloudFiles.format", "json")
                         .option("cloudFiles.schemaLocation", "/Volumes/gizmobox/landing/operational_data/customers_autoloader/_schema")
                         .option("cloudFiles.inferColumnTypes", "true")
                         .option("cloudFiles.schemaHints", "date_of_birth DATE, member_since DATE, created_timestamp TIMESTAMP")
                         .load("/Volumes/gizmobox/landing/operational_data/customers_autoloader/")
)

## 2. Add lineage columns
Record the source `file_path` and `ingestion_date` for traceability.

In [0]:
from pyspark.sql.functions import current_timestamp, col

customers_transformed_df = (
                                customers_df.withColumn("file_path", col("_metadata.file_path"))
                                            .withColumn("ingestion_date", current_timestamp())
)

## 3. Write to the Bronze Delta table
Checkpointed streaming write. Auto Loader remembers which files it has already processed, so reruns only pick up new arrivals.

In [0]:
streaming_query = (
                    customers_transformed_df.writeStream
                        .format("delta")
                        .option("checkpointLocation", "/Volumes/gizmobox/landing/operational_data/customers_autoloader/_checkpoint_stream")
                        .toTable("gizmobox.bronze.customers_autoloader")
)

## 4. Stop the stream

In [0]:
streaming_query.stop()

## 5. Verify the results

In [0]:
%sql
SELECT * FROM gizmobox.bronze.customers_autoloader;